# Carga

In [160]:
import requests
import pandas as pd
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots


url = "https://mindicador.cl/api/dolar/2026"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2026.csv", index=False)

In [118]:
origen = Path("../datasets/bencina_en_linea_comprimido")
destino = Path("../datasets/bencina_en_linea")
destino.mkdir(exist_ok=True)
for archivo in origen.glob("*.csv.bz2"):
    df = pd.read_csv(
        archivo,
        compression="bz2",
        encoding="latin1",
        engine="python",
        sep=None
    )
    salida = destino / archivo.name.replace(".csv.bz2", ".parquet")
    df.to_parquet(salida, index=False)

KeyboardInterrupt: 

# Limpieza

In [ ]:
df_2012 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2012.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad Herrera Bravo Ltda,COPEC,Avenida IrarrÃ¡zaval,5277.0,ÃuÃ±oa,Metropolitana,759,2012-01-01,Gasolina 93,"-33,4540062109273","-70,57525992393493"
1,pb630302,Distribuidora Dagnino Giacobbe y CÃ­a. Ltda. ...,Sin Bandera,Camino a Codegua Km,1.0,Chimbarongo,Gral. Bernardo O'Higgins,612,2012-12-13,Petroleo Diesel,"-34,71739081954427","-71,02941691875458"
2,co120501,SOCIEDAD COMERCIAL IBAÃ¯Â¿Â½EZ Y NEGRON LTDA.,COPEC,"PANAMERICANA NORTE KM 1750, EX S. V",0.0,Pozo Almonte,TarapacÃ¡,777,2012-12-06,Gasolina 95,"-21,09225805724834","-69,5928230881691"
3,te1312301,Gilberto Zamorano Vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,"-33,44420837761395","-70,59720039367676"
4,co1313201,PATRICIO REYES INFANTE Y CIA LTDA,COPEC,Av. Vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,"-33,38974288164123","-70,57051241397858"
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa Rosa,25.0,Puente Alto,Metropolitana,750,2012-06-29,Gasolina 93,"-33,61835394119413","-70,62618434429169"
6,sh510601,COMERCIALIZADORA DE COMBUSTIBLES ENERGIAS Y OT...,SHELL,Avda. Los Carrera,620.0,QuilpuÃ©,ValparaÃ­so,846,2012-08-16,Gasolina 97,"-33,04439263678347","-71,46056592464447"
7,co830501,Administradora de Ventas al Detalle Ltda. ...,COPEC,LASTARRIA,11.0,MulchÃ©n,BÃ­o BÃ­o,623,2012-06-11,Petroleo Diesel,"-37,71319890348465","-72,24748313426971"
8,co1311902,Villanueva y Clark Limitada,COPEC,Alberto Llona,640.0,MaipÃº,Metropolitana,782,2012-11-22,Gasolina 97,"-33,52131677600925","-70,75603008270264"
9,te1340106,ALONSO SPA,Sin Bandera,Panamericana Sur Km 17,12667.0,San Bernardo,Metropolitana,820,2012-03-15,Gasolina 93,"-33,56301455104863","-70,71177363395691"


In [ ]:
df_2012.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251917 entries, 0 to 251916
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   251917 non-null  object 
 1   razon_social         251917 non-null  object 
 2   distribuidor         251917 non-null  object 
 3   direccion_calle      251917 non-null  object 
 4   direccion_numero     251917 non-null  float64
 5   comuna               251917 non-null  object 
 6   region               251917 non-null  object 
 7   precio               251917 non-null  int64  
 8   fecha_actualizacion  251917 non-null  object 
 9   combustible          251917 non-null  object 
 10  latitud              251917 non-null  object 
 11  longitud             251917 non-null  object 
dtypes: float64(1), int64(1), object(10)
memory usage: 23.1+ MB


## Latitud - Longitud

In [ ]:
def limpiar_cordenada(coordenada):
    if ',' in coordenada:
        coordenada = coordenada.replace(",", ".")
    
    cardinalidad = ["N", "S", "E", "W", "n", "s", "e", "w"]
    for cardinal in cardinalidad:
        if coordenada.endswith(cardinal):
            coordenada = coordenada.replace(cardinal, "")
            break
    
    return coordenada

df_2012["latitud"] = df_2012["latitud"].apply(limpiar_cordenada).astype(float)
df_2012["longitud"] = df_2012["longitud"].apply(limpiar_cordenada).astype(float)

df_2012.tail()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
251912,co630103,COMERCIAL H Y M LTDA,COPEC,AV. B. OHIGGINS ESQ. M. VELASCO,0.0,San Fernando,Gral. Bernardo O'Higgins,825,2012-06-28,Gasolina 97,-34.584062,-70.983728
251913,co1320101,Arriagada y Mora Limitada,COPEC,Av. Concha y Toro,316.0,Puente Alto,Metropolitana,596,2012-12-20,Petroleo Diesel,-33.616764,-70.574214
251914,pe1410101,Sociedad Jorge Contreras CaÃ±as y Cia. Ltda.,PETROBRAS,Picarte,1027.0,Valdivia,Los RÃ­Â­os,645,2012-11-08,Petroleo Diesel,-39.817165,-73.234777
251915,te1312904,COMERCIAL E INVERSIONES DELGADO HERMANOS & CIA...,SHELL,Santa Rosa,2700.0,San JoaquÃ­n,Metropolitana,576,2012-12-19,Petroleo Diesel,-33.481286,-70.641280
251916,co1311302,Inversiones Cossio y Hutt Ltda.,COPEC,Avenida Ossa,591.0,La Reina,Metropolitana,853,2012-08-31,Gasolina 97,-33.448246,-70.571183


## Fecha

In [ ]:
df_2012['fecha_actualizacion'] = pd.to_datetime(df_2012['fecha_actualizacion'], errors='coerce')
df_2012.head()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad Herrera Bravo Ltda,COPEC,Avenida IrarrÃ¡zaval,5277.0,ÃuÃ±oa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora Dagnino Giacobbe y CÃ­a. Ltda. ...,Sin Bandera,Camino a Codegua Km,1.0,Chimbarongo,Gral. Bernardo O'Higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,SOCIEDAD COMERCIAL IBAÃ¯Â¿Â½EZ Y NEGRON LTDA.,COPEC,"PANAMERICANA NORTE KM 1750, EX S. V",0.0,Pozo Almonte,TarapacÃ¡,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto Zamorano Vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,PATRICIO REYES INFANTE Y CIA LTDA,COPEC,Av. Vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512


## Texto

In [ ]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    try:
        valor = valor.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        valor = valor
    return valor.capitalize()

df_2012['razon_social'] = df_2012['razon_social'].apply(limpiar_texto)
df_2012['comuna'] = df_2012['comuna'].apply(limpiar_texto)
df_2012['direccion_calle'] = df_2012['direccion_calle'].apply(limpiar_texto)
df_2012['region'] = df_2012['region'].apply(limpiar_texto)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277.0,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1.0,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0.0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa rosa,25.0,Puente alto,Metropolitana,750,2012-06-29,Gasolina 93,-33.618354,-70.626184
6,sh510601,Comercializadora de combustibles energias y ot...,SHELL,Avda. los carrera,620.0,Quilpué,Valparaíso,846,2012-08-16,Gasolina 97,-33.044393,-71.460566
7,co830501,Administradora de ventas al detalle ltda.,COPEC,Lastarria,11.0,Mulchén,Bío bío,623,2012-06-11,Petroleo Diesel,-37.713199,-72.247483
8,co1311902,Villanueva y clark limitada,COPEC,Alberto llona,640.0,Maipú,Metropolitana,782,2012-11-22,Gasolina 97,-33.521317,-70.756030
9,te1340106,Alonso spa,Sin Bandera,Panamericana sur km 17,12667.0,San bernardo,Metropolitana,820,2012-03-15,Gasolina 93,-33.563015,-70.711774


## direccion_numero

In [ ]:
df_2012['direccion_numero'] = df_2012['direccion_numero'].astype(int)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa rosa,25,Puente alto,Metropolitana,750,2012-06-29,Gasolina 93,-33.618354,-70.626184
6,sh510601,Comercializadora de combustibles energias y ot...,SHELL,Avda. los carrera,620,Quilpué,Valparaíso,846,2012-08-16,Gasolina 97,-33.044393,-71.460566
7,co830501,Administradora de ventas al detalle ltda.,COPEC,Lastarria,11,Mulchén,Bío bío,623,2012-06-11,Petroleo Diesel,-37.713199,-72.247483
8,co1311902,Villanueva y clark limitada,COPEC,Alberto llona,640,Maipú,Metropolitana,782,2012-11-22,Gasolina 97,-33.521317,-70.756030
9,te1340106,Alonso spa,Sin Bandera,Panamericana sur km 17,12667,San bernardo,Metropolitana,820,2012-03-15,Gasolina 93,-33.563015,-70.711774


In [ ]:
df_2012['razon_social'].astype(str)
df_2012['comuna'].astype(str)
df_2012['direccion_calle'].astype(str)
df_2012['region'].astype(str)
df_2012['distribuidor'].astype(str)
df_2012['combustible'].astype(str)
df_2012.head()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512


In [ ]:
df_2012.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251917 entries, 0 to 251916
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   id                   251917 non-null  object        
 1   razon_social         251917 non-null  object        
 2   distribuidor         251917 non-null  object        
 3   direccion_calle      251917 non-null  object        
 4   direccion_numero     251917 non-null  int64         
 5   comuna               251917 non-null  object        
 6   region               251917 non-null  object        
 7   precio               251917 non-null  int64         
 8   fecha_actualizacion  251917 non-null  datetime64[ns]
 9   combustible          251917 non-null  object        
 10  latitud              251917 non-null  float64       
 11  longitud             251917 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(7)
memory usage: 23.1

In [119]:
df_2013 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2013.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2014 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2014.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2015 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2015.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2016 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2016.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)   
df_2017 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2017.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2018 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2018.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2019 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2019.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2020 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2020.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)      
df_2021 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2021.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2022 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2022.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)



In [120]:
df_2023 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2023.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)  
df_2024 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2024.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2025 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2025.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2026 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2026.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)

In [121]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026]:
    print(df.columns.tolist())

['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud'

In [122]:
print(df_2023.columns.tolist())

['codigo', 'razon_social', 'distribuidor', 'direccion', 'latitud', 'longitud', 'nom_comuna', 'nom_region', 'combustible', 'precio', 'unidad_cobro', 'atencion', 'fecha_actualizacion', 'hora_actualizacion', 'es_electrolinera', 'es_gasolinera']


In [123]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022]:
    df['latitud'] = df['latitud'].apply(limpiar_cordenada).astype(float)
    df['longitud'] = df['longitud'].apply(limpiar_cordenada).astype(float)
    df['fecha_actualizacion'] = pd.to_datetime(df['fecha_actualizacion'], errors='coerce')
    df['razon_social'] = df['razon_social'].apply(limpiar_texto)
    df['comuna'] = df['comuna'].apply(limpiar_texto)
    df['direccion_calle'] = df['direccion_calle'].apply(limpiar_texto)
    df['region'] = df['region'].apply(limpiar_texto)
    df['direccion_numero'] = df['direccion_numero'].astype(int)  

In [124]:
cols_renombrar = {
    'codigo': 'id',
    'direccion': 'direccion_calle',
    'nom_comuna': 'comuna',
    'nom_region': 'region'
}

for df in [df_2023, df_2024, df_2025, df_2026]:
    df.rename(columns=cols_renombrar, inplace=True)
    df.drop(columns=['hora_actualizacion', 'unidad_cobro', 'atencion'], inplace=True)
    df.drop_duplicates(inplace=True)
    df['es_electrolinera'] = df['es_electrolinera'].astype(bool)
    df['es_gasolinera'] = df['es_gasolinera'].astype(bool)
    df['latitud'] = df['latitud'].apply(limpiar_cordenada).astype(float)
    df['longitud'] = df['longitud'].apply(limpiar_cordenada).astype(float)
    df['fecha_actualizacion'] = pd.to_datetime(df['fecha_actualizacion'], errors='coerce')
    df['razon_social'] = df['razon_social'].apply(limpiar_texto)
    df['comuna'] = df['comuna'].apply(limpiar_texto)
    df['direccion_calle'] = df['direccion_calle'].apply(limpiar_texto)
    df['region'] = df['region'].apply(limpiar_texto)
    df['precio'] = df['precio'].astype(int)

In [125]:
df_2025.head(10)

,id,razon_social,distribuidor,direccion_calle,latitud,longitud,comuna,region,combustible,precio,fecha_actualizacion,es_electrolinera,es_gasolinera
0,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1282,2024-12-20,False,True
1,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1310,2025-01-02,False,True
2,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1315,2025-01-08,False,True
3,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1344,2025-01-09,False,True
4,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1343,2025-01-23,False,True
5,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1372,2025-01-31,False,True
6,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1371,2025-02-14,False,True
7,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1349,2025-02-20,False,True
8,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1348,2025-02-27,False,True
9,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1320,2025-03-13,False,True


## Duplicados

In [126]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026]:
    df.sort_values('fecha_actualizacion', inplace=True)
    df.drop_duplicates(subset=['id', 'razon_social','combustible'], keep='last', inplace=True)

In [127]:
df_2025.head()

,id,razon_social,distribuidor,direccion_calle,latitud,longitud,comuna,region,combustible,precio,fecha_actualizacion,es_electrolinera,es_gasolinera
222909,co1312001,Inversiones pargar ltda,COPEC,Avenida irarrázaval 1102,-33.453098,-70.619141,Ñuñoa,Metropolitana de santiago,GNC,100,2012-01-12,False,True
98073,pb740103,Inversiones hn spa,HN,Avenida presidente ibañez 1175,-35.842647,-71.581807,Linares,Del maule,97,700,2012-03-22,False,True
29088,co410104,Kaptol ltda,COPEC,Balmaceda 1839,-29.917805,-71.253902,La serena,Coquimbo,GLP,0,2012-04-10,False,True
213729,co1312801,Granese y urrutia ltda.,COPEC,Américo vespucio esq. lo boza 2001,-33.386357,-70.758981,Renca,Metropolitana de santiago,KE,633,2012-04-12,False,True
100407,hn740701,Inversiones hn spa,HN,Abate molina 160,-35.676128,-71.743874,Villa alegre,Del maule,GNC,563,2012-05-02,False,True


# Evolución temporal del precio (solo bencinas)

¿Cómo ha evolucionado el precio promedio mensual de la bencina 93 y 97 entre 2012 y 2026?

In [139]:
df_total = pd.concat([
    df_2012, df_2013, df_2014, df_2015, df_2016, df_2017,
    df_2018, df_2019, df_2020, df_2021, df_2022, df_2023,
    df_2024, df_2025, df_2026
], ignore_index=True)

In [140]:
print(df_total['combustible'].unique())

['Gasolina 93' 'Petroleo Diesel' 'Gasolina 95' 'Gasolina 97' 'Kerosene'
 'GLP Vehicular' 'GNC' 'Electricidad' '97' 'GLP' 'KE' '95' 'DI' '93' 'ADI'
 'A93' 'A95' 'A97' 'AKE']


Todos los nombres al mismo valor

In [141]:
mapa_combustible = {
    'Gasolina 93': 'Gasolina 93', '93': 'Gasolina 93', 'A93': 'Gasolina 93',
    'Gasolina 95': 'Gasolina 95', '95': 'Gasolina 95', 'A95': 'Gasolina 95',
    'Gasolina 97': 'Gasolina 97', '97': 'Gasolina 97', 'A97': 'Gasolina 97',
    'Petroleo Diesel': 'Diesel',   'DI': 'Diesel',     'ADI': 'Diesel',
}
df_total['anio'] = df_total['fecha_actualizacion'].dt.year
df_total['mes'] = df_total['fecha_actualizacion'].dt.month
df_total['combustible_limpio'] = df_total['combustible'].map(mapa_combustible)
df_filtrado = df_total[df_total['combustible_limpio'].notna()].copy()
precio_anual = df_filtrado.groupby(['anio', 'combustible_limpio'])['precio'].mean().reset_index()

print(precio_anual)

    anio combustible_limpio       precio
0   2012             Diesel   618.866569
1   2012        Gasolina 93   789.471759
2   2012        Gasolina 95   809.489635
3   2012        Gasolina 97   829.925527
4   2013             Diesel   645.266301
5   2013        Gasolina 93   790.737461
6   2013        Gasolina 95   810.571692
7   2013        Gasolina 97   798.768207
8   2014             Diesel   578.501847
9   2014        Gasolina 93   728.316186
10  2014        Gasolina 95   755.580846
11  2014        Gasolina 97   771.334989
12  2015             Diesel   452.340012
13  2015        Gasolina 93   670.965807
14  2015        Gasolina 95   711.697910
15  2015        Gasolina 97   750.896978
16  2016             Diesel   486.567981
17  2016        Gasolina 93   703.746098
18  2016        Gasolina 95   744.064439
19  2016        Gasolina 97   768.923235
20  2017             Diesel   542.450000
21  2017        Gasolina 93   757.131894
22  2017        Gasolina 95   780.559474
23  2017        

In [142]:
fig = go.Figure()
for combustible in precio_anual['combustible_limpio'].unique():
    df_comb = precio_anual[precio_anual['combustible_limpio'] == combustible]
    fig.add_trace(go.Scatter(
        x=df_comb['anio'],
        y=df_comb['precio'],
        mode='lines+markers',
        name=combustible
    ))

fig.update_layout(
    title='Evolución del precio promedio de combustibles en Chile (2012-2026)',
    xaxis_title='Año',
    yaxis_title='Precio promedio (CLP)',
    height=500
)

fig.show()

El gráfico muestra la evolución del precio promedio anual de los cuatro combustibles principales en Chile entre 2012 y 2026.

Se observan tres períodos claramente distintos:

*   2012-2020: Los precios se mantienen relativamente estables, oscilando entre 600 y 900 CLP. Destaca una caída pronunciada entre 2014 y 2016, explicada por el desplome del precio internacional del petróleo crudo en ese período.

*   2020-2022: Se produce el alza más abrupta del período, donde todos los combustibles casi duplican su precio. Esto coincide con la recuperación post-pandemia y el conflicto entre Rusia y Ucrania en 2022, que generó un shock en el mercado energético mundial.

*   2022-2026: Los precios se estabilizan entre 2022 y 2025, pero en 2026 vuelven a subir con fuerza, superando los máximos históricos previos.

# Variación por región o distribuidora

¿Copec cobra más que Shell? ¿La Región Metropolitana es más cara que el norte?

El precio de los combustibles no es uniforme en todo el país ni entre todas las marcas. Existen dos factores que podrían explicar diferencias sistemáticas en los precios: la ubicación geográfica de la estación y la distribuidora que la opera.

Por el lado geográfico, las regiones más alejadas de los centros de distribución podrían enfrentar mayores costos de transporte, lo que se puede traducir en precios más altos para el consumidor final. Por el lado de las distribuidoras, marcas con mayor presencia y reconocimiento podrían cobrar un precio diferente al de estaciones sin bandera o de menor presencia en el mercado.

Identificar estas diferencias es relevante porque permite evaluar si el mercado de combustibles presenta patrones de precios que podrían indicar falta de competencia en ciertas zonas o entre ciertas marcas, lo cual constituye el tipo de evidencia que organismos reguladores como la FNE necesitan para investigar posibles prácticas anticompetitivas.

In [143]:
print("Regiones:")
print(df_filtrado['region'].unique())
print()
print("Distribuidoras:")
print(df_filtrado['distribuidor'].unique())

Regiones:
['Metropolitana' "Gral. bernardo o'higgins" 'Tarapacá' 'Valparaíso'
 'Bío bío' 'Los rí\xados' 'Araucanía' 'Maule'
 'Aysén gral. c. ibáñez del campo' 'Ñuble' 'Coquimbo' 'Antofagasta'
 'Arica y parinacota' 'Magallanes y la antártida chilena' 'Los lagos'
 'Atacama' 'Del maule' 'Metropolitana de santiago'
 'Magallanes y de la antártica chilena'
 'Del libertador gral. bernardo o’higgins' 'Del biobío' 'De la araucanía'
 'Aysén del gral. carlos ibáñez del campo' 'De los lagos' 'De los ríos']

Distribuidoras:
['COPEC' 'Sin Bandera' 'SHELL' 'PETROBRAS' 'SELK' 'COMBUSTIBLES JCD'
 'COMBUSTIBLES ANLOA' 'APEX' 'SOCORRO' 'TERPEL' 'HN' 'CUSTOM SERVICE'
 'JVL COMBUSTIBLES' 'FACAZ' 'SUAREZ COMBUSTIBLES' 'HOLA!'
 'COMBUSTIBLES AMADE' 'Combustibles Endless.com' 'AUTOGASCO'
 'Combustibles JSP' 'SIN BANDERA' 'EL HUIQUE' 'SESA' 'SERVICENTRO LEAL'
 'SURENERGY' 'Aire' 'CNC COMBUSTIBLES' 'JLC' 'Punto Sur'
 'Del Sol Combustibles' 'DELPA' 'PETRONEXT' 'CAVE' 'COMERCIAL MAQUI'
 'ECOIL' 'Infinia Combustib

## Problemas con regiones duplicadas

In [144]:
mapa_regiones = {
    'Metropolitana': 'Metropolitana',
    'Metropolitana de santiago': 'Metropolitana',
    "Gral. bernardo o'higgins": "O'Higgins",
    "Del libertador gral. bernardo o'higgins": "O'Higgins",
    'Tarapacá': 'Tarapacá',
    'Valparaíso': 'Valparaíso',
    'Bío bío': 'Biobío',
    'Del biobío': 'Biobío',
    'Los rí\xados': 'Los Ríos',
    'De los ríos': 'Los Ríos',
    'Araucanía': 'Araucanía',
    'De la araucanía': 'Araucanía',
    'Maule': 'Maule',
    'Del maule': 'Maule',
    'Aysén gral. c. ibáñez del campo': 'Aysén',
    'Aysén del gral. carlos ibáñez del campo': 'Aysén',
    'Ñuble': 'Ñuble',
    'Coquimbo': 'Coquimbo',
    'Antofagasta': 'Antofagasta',
    'Arica y parinacota': 'Arica y Parinacota',
    'Magallanes y la antártida chilena': 'Magallanes',
    'Magallanes y de la antártica chilena': 'Magallanes',
    'Los lagos': 'Los Lagos',
    'De los lagos': 'Los Lagos',
    'Atacama': 'Atacama',
}

df_filtrado['region_limpia'] = df_filtrado['region'].map(mapa_regiones)

**Sin bandera** : la bencinera no pertenece a ninguna marca grande. Es una estación independiente que puede comprar combustible donde quiera y venderlo bajo su propio nombre o sin nombre.

En los datos aparece escrito de dos formas distintas:

*   'Sin Bandera' — así viene en los años viejos
*   'SIN BANDERA' — así viene en los años nuevos

In [146]:
df_filtrado['distribuidor_limpio'] = df_filtrado['distribuidor']
df_filtrado.loc[df_filtrado['distribuidor'] == 'SIN BANDERA', 'distribuidor_limpio'] = 'Sin Bandera'
df_filtrado.columns

Index(['id', 'razon_social', 'distribuidor', 'direccion_calle',
       'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion',
       'combustible', 'latitud', 'longitud', 'es_electrolinera',
       'es_gasolinera', 'anio', 'mes', 'combustible_limpio', 'region_limpia',
       'distribuidor_limpio'],
      dtype='object')

Se crea una nueva columna llamada distribuidor_limpio que es una copia exacta de distribuidor. Lo hacemos así para no modificar la columna original distribuidor, así reemplazamos solo 'SIN BANDERA' por 'Sin Bandera' en la columna nueva, sin afectar la original.

In [147]:
distribuidoras_grandes = ['COPEC', 'SHELL', 'PETROBRAS', 'ENEX', 'TERPEL', 'Sin Bandera']
df_dist = df_filtrado[df_filtrado['distribuidor_limpio'].isin(distribuidoras_grandes)]
print("Distribuidoras:", df_dist['distribuidor_limpio'].unique())

Distribuidoras: ['COPEC' 'Sin Bandera' 'SHELL' 'PETROBRAS' 'TERPEL' 'ENEX']


### Precio promedio por región

In [148]:
precio_region = df_filtrado.groupby('region_limpia')['precio'].mean().reset_index()
precio_region = precio_region.sort_values('precio', ascending=True)

fig1 = go.Figure(go.Bar(
    x=precio_region['precio'],
    y=precio_region['region_limpia'],
    orientation='h',
    marker_color='steelblue',
    text=precio_region['precio'].round(0),
    textposition='outside'
))

fig1.update_layout(
    title='Precio promedio de combustibles por región (2012-2026)',
    xaxis_title='Precio promedio (CLP)',
    yaxis_title='Región',
    height=500
)

fig1.show()

Las diferencias de precio entre regiones son relativamente pequeñas, oscilando entre 778 CLP en O'Higgins y 900 CLP en Los Ríos. Sin embargo, se observa un patrón interesante respecto a que las regiones más baratas son las del centro del país (O'Higgins, Metropolitana, Maule), mientras que las más caras son regiones extremas o aisladas como Los Ríos, Aysén y Antofagasta. Esto es consistente con la hipótesis de que los costos de transporte y distribución encarecen el combustible en zonas alejadas de los centros de abastecimiento.

### Precio promedio por distribuidora

In [149]:
precio_dist = df_dist.groupby('distribuidor_limpio')['precio'].mean().reset_index()
precio_dist = precio_dist.sort_values('precio', ascending=True)

fig2 = go.Figure(go.Bar(
    x=precio_dist['precio'],
    y=precio_dist['distribuidor_limpio'],
    orientation='h',
    marker_color='steelblue',
    text=precio_dist['precio'].round(0),
    textposition='outside'
))

fig2.update_layout(
    title='Precio promedio de combustibles por distribuidora (2012-2026)',
    xaxis_title='Precio promedio (CLP)',
    yaxis_title='Distribuidora',
    height=400
)

fig2.show()

Copec y Shell son las distribuidoras más caras, con precios promedio de 834 y 829 CLP respectivamente, mientras que Terpel es la más barata con 755 CLP. Las estaciones Sin Bandera y Enex se ubican en un rango intermedio con 788 CLP. Esto sugiere que las marcas con mayor reconocimiento y presencia nacional cobran un precio mayor respecto a competidores más pequeños, lo que podría reflejar tanto diferencias en calidad percibida como menor presión competitiva en las zonas donde operan.

In [150]:
grandes = ['COPEC', 'SHELL', 'PETROBRAS', 'ENEX', 'TERPEL']
sin_bandera = ['Sin Bandera', 'SIN BANDERA']

def clasificar_distribuidor(distribuidor):
    if distribuidor in grandes:
        return 'Grande'
    elif distribuidor in sin_bandera:
        return 'Sin Bandera'
    else:
        return 'Independiente'

In [151]:
df_filtrado['tipo_distribuidor'] = df_filtrado['distribuidor'].apply(clasificar_distribuidor)

print(df_filtrado['tipo_distribuidor'].value_counts())

tipo_distribuidor
Grande           281283
Independiente     25499
Sin Bandera       13635
Name: count, dtype: int64


In [152]:
df_filtrado.columns

Index(['id', 'razon_social', 'distribuidor', 'direccion_calle',
       'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion',
       'combustible', 'latitud', 'longitud', 'es_electrolinera',
       'es_gasolinera', 'anio', 'mes', 'combustible_limpio', 'region_limpia',
       'distribuidor_limpio', 'tipo_distribuidor'],
      dtype='object')

In [153]:
composicion = df_filtrado.groupby(['region_limpia', 'tipo_distribuidor']).size().reset_index(name='cantidad')
total_por_region = composicion.groupby('region_limpia')['cantidad'].transform('sum')
composicion['porcentaje'] = composicion['cantidad'] / total_por_region * 100

orden = composicion[composicion['tipo_distribuidor'] == 'Grande'].sort_values('porcentaje', ascending=True)['region_limpia']

fig = go.Figure()

colores = {'Grande': '#2ecc71', 'Sin Bandera': '#e74c3c', 'Independiente': '#f39c12'}

for tipo in ['Grande', 'Sin Bandera', 'Independiente']:
    df_tipo = composicion[composicion['tipo_distribuidor'] == tipo].set_index('region_limpia')
    fig.add_trace(go.Bar(
        y=orden,
        x=[df_tipo.loc[r, 'porcentaje'] if r in df_tipo.index else 0 for r in orden],
        name=tipo,
        orientation='h',
        marker_color=colores[tipo]
    ))

fig.update_layout(
    barmode='stack',
    title='Composición de distribuidoras por región (2012-2026)',
    xaxis_title='Porcentaje (%)',
    yaxis_title='Región',
    height=500
)

fig.show()

El gráfico muestra qué proporción de los registros de cada región corresponde a distribuidoras grandes, sin bandera e independientes.
Se observa que en prácticamente todas las regiones las distribuidoras grandes dominan ampliamente, superando el 80% en la mayoría de los casos. Sin embargo, hay diferencias notables entre regiones.

Las regiones del norte como Antofagasta, Atacama y Tarapacá tienen la mayor concentración de distribuidoras grandes, con muy poca presencia de independientes. Esto puede explicarse porque en zonas mineras y de alto tráfico industrial, las grandes marcas tienen más incentivos para instalarse.
En cambio, las regiones del centro-sur como Maule, Ñuble, Araucanía y O'Higgins muestran una presencia significativamente mayor de distribuidoras independientes y sin bandera. Esto sugiere que en zonas más rurales o agrícolas, donde el mercado es menos atractivo para las grandes marcas, proliferan más los actores independientes.

# Relación dólar vs precio bencina

In [155]:
url = "https://mindicador.cl/api/dolar/2012"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2012.csv", index=False)

url = "https://mindicador.cl/api/dolar/2013"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2013.csv", index=False)

url = "https://mindicador.cl/api/dolar/2014"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2014.csv", index=False)

url = "https://mindicador.cl/api/dolar/2015"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2015.csv", index=False)

url = "https://mindicador.cl/api/dolar/2016"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])    
df.to_csv("../datasets/dolar_2016.csv", index=False)

url = "https://mindicador.cl/api/dolar/2017"  
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2017.csv", index=False)

url = "https://mindicador.cl/api/dolar/2018"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2018.csv", index=False)

url = "https://mindicador.cl/api/dolar/2019"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2019.csv", index=False)

url = "https://mindicador.cl/api/dolar/2020"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2020.csv", index=False)

url = "https://mindicador.cl/api/dolar/2021"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2021.csv", index=False)

url = "https://mindicador.cl/api/dolar/2022"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2022.csv", index=False)

url = "https://mindicador.cl/api/dolar/2023"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2023.csv", index=False)

url = "https://mindicador.cl/api/dolar/2024"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2024.csv", index=False)

url = "https://mindicador.cl/api/dolar/2025"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2025.csv", index=False)

url = "https://mindicador.cl/api/dolar/2026"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2026.csv", index=False)

In [156]:
df_dolar_2012 = pd.read_csv('../datasets/dolar_2012.csv')
df_dolar_2013 = pd.read_csv('../datasets/dolar_2013.csv')
df_dolar_2014 = pd.read_csv('../datasets/dolar_2014.csv')
df_dolar_2015 = pd.read_csv('../datasets/dolar_2015.csv')
df_dolar_2016 = pd.read_csv('../datasets/dolar_2016.csv')
df_dolar_2017 = pd.read_csv('../datasets/dolar_2017.csv')
df_dolar_2018 = pd.read_csv('../datasets/dolar_2018.csv')
df_dolar_2019 = pd.read_csv('../datasets/dolar_2019.csv')
df_dolar_2020 = pd.read_csv('../datasets/dolar_2020.csv')
df_dolar_2021 = pd.read_csv('../datasets/dolar_2021.csv')
df_dolar_2022 = pd.read_csv('../datasets/dolar_2022.csv')
df_dolar_2023 = pd.read_csv('../datasets/dolar_2023.csv')
df_dolar_2024 = pd.read_csv('../datasets/dolar_2024.csv')
df_dolar_2025 = pd.read_csv('../datasets/dolar_2025.csv')
df_dolar_2026 = pd.read_csv('../datasets/dolar_2026.csv')


df_dolar = pd.concat([
    df_dolar_2012, df_dolar_2013, df_dolar_2014, df_dolar_2015,
    df_dolar_2016, df_dolar_2017, df_dolar_2018, df_dolar_2019,
    df_dolar_2020, df_dolar_2021, df_dolar_2022, df_dolar_2023,
    df_dolar_2024, df_dolar_2025, df_dolar_2026
], ignore_index=True)

df_dolar['fecha'] = pd.to_datetime(df_dolar['fecha'])
df_dolar['anio'] = df_dolar['fecha'].dt.year

print(f"Total de filas: {len(df_dolar)}")
print(df_dolar.head(5))

Total de filas: 3587
                      fecha   valor  anio
0 2012-12-28 03:00:00+00:00  478.60  2012
1 2012-12-27 03:00:00+00:00  479.09  2012
2 2012-12-26 03:00:00+00:00  479.39  2012
3 2012-12-24 03:00:00+00:00  476.40  2012
4 2012-12-21 03:00:00+00:00  475.02  2012


Promedio anual del dolar

In [157]:
dolar_anual = df_dolar.groupby('anio')['valor'].mean().reset_index()
dolar_anual.columns = ['anio', 'dolar_promedio']

Promedio anual del precio (solo de la 93 para resumir un poquito)

In [158]:
bencina_anual = df_filtrado[df_filtrado['combustible_limpio'] == 'Gasolina 93'].groupby('anio')['precio'].mean().reset_index()
bencina_anual.columns = ['anio', 'precio_promedio']

In [159]:
df_relacion = pd.merge(bencina_anual, dolar_anual, on='anio')
print(df_relacion)

    anio  precio_promedio  dolar_promedio
0   2012       789.471759      486.746559
1   2013       790.737461      494.995161
2   2014       728.316186      570.005904
3   2015       670.965807      654.249000
4   2016       703.746098      676.832421
5   2017       757.131894      649.328785
6   2018       800.369449      640.290772
7   2019       840.831879      702.631048
8   2020       762.155867      792.221833
9   2021      1010.141122      759.272840
10  2022      1285.776888      872.113865
11  2023      1291.986673      839.073401
12  2024      1270.833954      943.582419
13  2025      1330.328495      951.641210
14  2026      1770.136459      890.624857


In [161]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=df_relacion['anio'], y=df_relacion['precio_promedio'],
               mode='lines+markers', name='Gasolina 93 (CLP)',
               line=dict(color='#e74c3c')),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df_relacion['anio'], y=df_relacion['dolar_promedio'],
               mode='lines+markers', name='Dólar (CLP)',
               line=dict(color='#2ecc71')),
    secondary_y=True
)

fig.update_layout(
    title='Evolución del precio de Gasolina 93 y el dólar (2012-2026)',
    xaxis_title='Año',
    height=500
)

fig.update_yaxes(title_text='Precio Gasolina 93 (CLP)', secondary_y=False)
fig.update_yaxes(title_text='Valor Dólar (CLP)', secondary_y=True)

fig.show()

El gráfico muestra la evolución conjunta del precio promedio de la Gasolina 93 y el valor del dólar en pesos chilenos entre 2012 y 2026, usando dos ejes distintos para poder comparar ambas variables en la misma escala visual.

Se observa que entre 2012 y 2020 ambas variables siguen trayectorias distintas, el dólar sube de forma sostenida mientras el precio de la bencina baja o se mantiene estable. Esto se explica principalmente por la caída del precio internacional del petróleo entre 2014 y 2016, que compensó el efecto del dólar alto.

A partir de 2020 las dos líneas comienzan a moverse en la misma dirección, subiendo con fuerza hasta 2022. Sin embargo, entre 2022 y 2025 el dólar sigue subiendo mientras la bencina se estabiliza, lo que podría explicarse por el efecto amortiguador del MEPCO.

En 2026 se produce un cruce interesante, el dólar baja levemente mientras la bencina sube con fuerza, alcanzando su máximo histórico.

### Evolución del precio de Gasolina 93 y el dólar con eventos geopolíticos (2012-2026)

In [163]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=df_relacion['anio'], y=df_relacion['precio_promedio'],
               mode='lines+markers', name='Gasolina 93 (CLP)',
               line=dict(color='#e74c3c')),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df_relacion['anio'], y=df_relacion['dolar_promedio'],
               mode='lines+markers', name='Dólar (CLP)',
               line=dict(color='#2ecc71')),
    secondary_y=True
)
fig.add_vline(x=2022, line_dash='dash', line_color='orange',
              annotation_text='Guerra Rusia-Ucrania', 
              annotation_position='top left')
fig.add_vline(x=2023, line_dash='dash', line_color='purple',
              annotation_text='Conflicto Israel-Hamas',
              annotation_position='top right')

# Línea vertical: Caída del petróleo
fig.add_vline(x=2014, line_dash='dash', line_color='steelblue',
              annotation_text='Caída precio petróleo',
              annotation_position='top right')

fig.update_layout(
    title='Evolución del precio de Gasolina 93 y el dólar (2012-2026)',
    xaxis_title='Año',
    height=500
)

fig.update_yaxes(title_text='Precio Gasolina 93 (CLP)', secondary_y=False)
fig.update_yaxes(title_text='Valor Dólar (CLP)', secondary_y=True)

fig.show()

# Referencias

[1] https://stackoverflow.com/questions/65315100/python-regular-expression-to-find-single-letter-cardinal-direction